# AutoGen — Runnable Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anacodicAI-labs/learn-notebooks/blob/main/ai-agents/6-2/autogen.ipynb) &nbsp; [View on GitHub](https://github.com/anacodicAI-labs/learn-notebooks/blob/main/ai-agents/6-2/autogen.ipynb)

Companion to: [AutoGen](https://anacodicai.org/learn/ai-agents/6-2/autogen)


## What you'll build

The **real `autogen-agentchat` library** (the modern AutoGen API — package
name `autogen_agentchat`, not the older `pyautogen`). Same trip-planning
example as the article's real-world comparison: instead of one orchestrator
calling specialists as tools, AutoGen puts the specialists in a **peer-to-peer
conversation**. We build:

1. A single `AssistantAgent` answering a question.
2. A `RoundRobinGroupChat` where a budget agent posts a price cap and a
   flights agent replies with options that fit it — agents actually
   conversing in turn, AutoGen's core differentiator versus a single
   prompt-in/answer-out chain.

**No API key needed** — `ReplayChatCompletionClient` is a real, built-in
AutoGen model client that plays back a fixed list of responses instead of
calling a network. `pip install autogen-agentchat autogen-ext` if running
locally. AutoGen's agents are `async`, so every call below is awaited.

### Setup

In [1]:
# %pip install -q autogen-agentchat autogen-ext
import asyncio

from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import MaxMessageTermination
from autogen_core import CancellationToken
from autogen_ext.models.replay import ReplayChatCompletionClient

print("real autogen-agentchat imported")

real autogen-agentchat imported


## Step 1 — a replay model client

`ReplayChatCompletionClient` takes a plain list of strings and returns them
in order, one per call — it satisfies AutoGen's real model-client interface,
so every agent built on top of it behaves exactly like it would with a real
`OpenAIChatCompletionClient`.

In [2]:
assistant_client = ReplayChatCompletionClient([
    "A 5-day trip to Barcelona for 4 people, flying out of Boston, "
    "in the shoulder season, usually runs $700-950 per person including flights and a hotel.",
])
print(type(assistant_client).__name__, "is a real autogen_ext model client")

ReplayChatCompletionClient is a real autogen_ext model client


## Step 2 — a single AssistantAgent

`on_messages` is the real per-turn entry point AutoGen agents expose. It
takes a list of messages and a `CancellationToken` (also real — used for
timeouts/cancellation in production, unused here) and returns a response
whose `.chat_message` is what the agent said.

In [3]:
trip_assistant = AssistantAgent("trip_assistant", model_client=assistant_client)

async def ask(agent, text):
    response = await agent.on_messages(
        [TextMessage(content=text, source="user")],
        cancellation_token=CancellationToken(),
    )
    return response.chat_message.content

answer = await ask(trip_assistant, "Roughly what would a 5-day Barcelona trip for 4 people cost?")
print("answer:", answer)

assert "Barcelona" in answer
print("\nassert passed: the agent answered using the replay client's canned response")

answer: A 5-day trip to Barcelona for 4 people, flying out of Boston, in the shoulder season, usually runs $700-950 per person including flights and a hotel.

assert passed: the agent answered using the replay client's canned response


## Step 3 — two agents that actually talk to each other

This is what a single LCEL chain or a single CrewAI agent can't do on its
own: a `RoundRobinGroupChat` hands the conversation from one agent to the
next automatically. `MaxMessageTermination(3)` stops after 3 messages —
the task, the budget agent's turn, and the flights agent's turn.

In [4]:
budget_client = ReplayChatCompletionClient([
    "Budget cap for this trip: $900 per person. Flights and hotel need to fit inside that.",
])
flights_client = ReplayChatCompletionClient([
    "Found a round-trip option for $420 per person, well under the cap.",
])

budget = AssistantAgent("budget", model_client=budget_client)
flights = AssistantAgent("flights", model_client=flights_client)

team = RoundRobinGroupChat(
    [budget, flights], termination_condition=MaxMessageTermination(3)
)
print("team built with agents:", [a.name for a in [budget, flights]])

team built with agents: ['budget', 'flights']


## Step 4 — run the team and read the transcript

Watch the `source` on each message: `user` (the task), then `budget`, then
`flights` — AutoGen routed the turn automatically, in the order the agents
were listed.

In [5]:
result = await team.run(task="Plan flights for the Barcelona trip.")

for message in result.messages:
    print(f"{message.source:10}: {message.content}")

assert result.messages[0].source == "user"
assert result.messages[1].source == "budget"
assert result.messages[2].source == "flights"
print("\nasserts passed: the group chat routed task -> budget -> flights automatically")

user      : Plan flights for the Barcelona trip.
budget    : Budget cap for this trip: $900 per person. Flights and hotel need to fit inside that.
flights   : Found a round-trip option for $420 per person, well under the cap.

asserts passed: the group chat routed task -> budget -> flights automatically


## Step 5 — the termination condition actually terminates

**Try it:** change `MaxMessageTermination(3)` to `MaxMessageTermination(2)`
and re-run Step 3+4 — the transcript stops after the budget agent's turn and
the flights agent never speaks, because the condition is checked after every
message, not just at the end.

In [6]:
print("number of messages in the transcript:", len(result.messages))
assert len(result.messages) == 3
print("assert passed: MaxMessageTermination(3) stopped the conversation at exactly 3 messages")

number of messages in the transcript: 3
assert passed: MaxMessageTermination(3) stopped the conversation at exactly 3 messages


## Try it yourself

- Add a third agent, `hotels`, that critiques the flights agent's pick
  against the group's preferred dates — exactly the peer-to-peer contrast
  the article draws against a single orchestrator calling specialists as
  tools. Increase `MaxMessageTermination` to `4` and watch
  `RoundRobinGroupChat` add it into the rotation with no other code changes.
- Replace `MaxMessageTermination` with a `TextMentionTermination("APPROVE")`
  — the conversation now stops as soon as any agent's message contains that
  word, regardless of turn count.
- Swap `ReplayChatCompletionClient` for
  `autogen_ext.models.openai.OpenAIChatCompletionClient` with a real key —
  the agents, the team, and the termination condition are all unchanged.